In [1]:
import pickle
from pathlib import Path

import pandas as pd

import params_scrap_syride as params
from rapport.report_data import prepare_report_df
from rapport.report_html import build_report

pd.set_option('display.max_columns', 1000)
pd.set_option('display.max_rows', 150)
pd.options.mode.chained_assignment = None

%load_ext autoreload
%autoreload 2


/var/folders/10/1m7q3nk502l57_kgv7_tyjch0000gp/T/ipykernel_10671/1803693274.py:12: Pandas4Warning: The 'mode.chained_assignment' option is deprecated. It has had no effect since pandas 3.0, which removed the SettingWithCopyWarning it controlled. Chained assignment now emits a ChainedAssignmentError warning instead; use warnings.filterwarnings with category=ChainedAssignmentError to silence or escalate it. This option will be removed in pandas 4.0.
  pd.options.mode.chained_assignment = None


In [2]:
# Paramètres
YEAR = 2026
# Pilotes à traiter ; None = tous les pilotes ayant volé dans l'année
PILOTS = None#["AdrienM"]
# Dossier de sortie relatif au dépôt (le notebook tourne depuis src/)
OUTPUT_DIR = Path.cwd().parent / "html_reports" / f"{YEAR}_html_reports"
SAVE_HTML = True
SAVE_PDF = False


In [3]:
# Chargement du pkl, puis préparation une seule fois (heure_decolage, flight_url, dates).
# compute_report détecte ce marquage et ne refait pas la préparation pour chaque pilote.
with open(params.pkl_path, "rb") as pkl_file:
    df = pickle.load(pkl_file)
df = prepare_report_df(df)

pilots = (
    PILOTS
    if PILOTS is not None
    else sorted(df.loc[df["date"].dt.year == YEAR, "pilote"].dropna().unique())
)
print(f"{df.shape[0]} vols, {len(pilots)} pilote(s) : {list(pilots)}")


10234 vols, 13 pilote(s) : ['AdrienM', 'Au_Phil_du_vent', 'DavidParatrucs79', 'Florent_Essar', 'JJvole', 'LiLi06', 'Thomas16', 'Tof Torsac', 'ToriVsUke', 'chami', 'j2f', 'jeanlouisksu', 'yanis']


In [4]:
# Rapport PDF (optionnel) : les dépendances PDF ne sont importées que si nécessaire
if SAVE_PDF:
    from report_functions import generate_pdf_report

    for pilot in pilots:
        generate_pdf_report(df=df, pilot=pilot, year=YEAR)


In [5]:
if SAVE_HTML:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    written = []

    for pilot in pilots:
        output = OUTPUT_DIR / f"{pilot}_rapport_vol_{YEAR}.html"
        try:
            build_report(
                df=df,
                pilot=pilot,
                year=YEAR,
                output=str(output),
                compare_years="auto",  # jusqu'à deux années précédentes disponibles
            )
        except ValueError as err:  # pilote sans vol cette année (ou aucun vol avant la date de coupure)
            print(f"Pas de rapport pour {pilot} : {err}")
            continue
        written.append(output)
        print(f"Rapport créé : {output.resolve()}")

    # Récapitulatif
    print(f"\n{len(written)} rapport(s) écrit(s) dans {OUTPUT_DIR.resolve()}")
    for path in written:
        size_ko = path.stat().st_size / 1024
        print(f"  {path.name} : {size_ko:,.0f} ko")
        if size_ko > 1024:
            print(f"  ATTENTION : {path.name} dépasse 1 Mo")


Rapport créé : /Users/Adrien/Documents/paramoteur/syride/analyze_traces/syride_traces/html_reports/2026_html_reports/AdrienM_rapport_vol_2026.html
Rapport créé : /Users/Adrien/Documents/paramoteur/syride/analyze_traces/syride_traces/html_reports/2026_html_reports/Au_Phil_du_vent_rapport_vol_2026.html
Rapport créé : /Users/Adrien/Documents/paramoteur/syride/analyze_traces/syride_traces/html_reports/2026_html_reports/DavidParatrucs79_rapport_vol_2026.html
Rapport créé : /Users/Adrien/Documents/paramoteur/syride/analyze_traces/syride_traces/html_reports/2026_html_reports/Florent_Essar_rapport_vol_2026.html
Rapport créé : /Users/Adrien/Documents/paramoteur/syride/analyze_traces/syride_traces/html_reports/2026_html_reports/JJvole_rapport_vol_2026.html
Rapport créé : /Users/Adrien/Documents/paramoteur/syride/analyze_traces/syride_traces/html_reports/2026_html_reports/LiLi06_rapport_vol_2026.html
Rapport créé : /Users/Adrien/Documents/paramoteur/syride/analyze_traces/syride_traces/html_report